# Loan Decision Engine using PD, LGD, EAD, and Expected Loss

## Objective

This notebook converts credit risk model outputs into a simplified loan decision engine.

The decision engine uses:

- Probability of Default (PD)
- Loss Given Default (LGD)
- Exposure at Default (EAD)
- Expected Loss
- Risk-adjusted return
- Risk band
- Risk score

The goal is to demonstrate how machine learning outputs can support practical lending decisions such as:

- Approve
- Manual Review
- Reject

In [5]:
import pandas as pd
import numpy as np 

pd.set_option('display.max_rows',100)
pd.set_option('display.max_columns',150)

## Import LGD, EAD, and Expected Loss Output

The dataset created in the previous notebook is imported.

This dataset already contains:

- borrower-level PD scores
- LGD estimates
- EAD values
- expected loss
- risk-adjusted return
- risk bands
- risk scores

These fields are used as inputs for the loan decision strategy.


In [6]:
df = pd.read_pickle('../data/lgd_ead_expected_loss_output.pkl')
print(df.shape)

df[[ "actual_default_flag", "pd_score", "lgd_estimate", "ead", "expected_loss", "risk_adjusted_return", "risk_band", "risk_score"]].head()

(40000, 152)


,actual_default_flag,pd_score,lgd_estimate,ead,expected_loss,risk_adjusted_return,risk_band,risk_score
0,1,0.495415,0.695648,12000,4135.613883,-1748.813883,High,505.0
1,0,0.504460,0.611491,20100,6200.299219,-3066.709219,High,496.0
2,0,0.301881,0.558952,8000,1349.895463,-458.695463,Low,698.0
3,0,0.685799,0.611491,23325,9781.568456,-6821.625956,Very High,314.0
4,0,0.739917,0.716095,21250,11259.330155,-6382.455155,Very High,260.0


## Review Key Credit Risk Metrics

The key risk-related columns are reviewed to confirm that the required decisioning variables are available.

Important fields:

- `actual_default_flag` → actual default outcome
- `pd_score` → predicted probability of default
- `lgd_estimate` → estimated loss given default
- `ead` → exposure at default
- `expected_loss` → estimated financial loss
- `risk_adjusted_return` → expected income after expected loss
- `risk_band` → borrower risk category
- `risk_score` → scorecard-style risk score

In [7]:
df[["pd_score", "lgd_estimate", "ead", "expected_loss", "risk_adjusted_return", "risk_band", "risk_score"]].describe()

,pd_score,lgd_estimate,ead,expected_loss,risk_adjusted_return,risk_score
count,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000
mean,0.438078,0.597925,14513.557500,4163.416110,-2160.433774,561.922650
std,0.215661,0.053360,8711.779018,3908.355002,2526.859363,215.661473
min,0.025880,0.531056,500.000000,33.600831,-17734.927705,32.000000
25%,0.263283,0.558952,8000.000000,1317.465334,-3214.107339,402.000000
50%,0.421281,0.611491,12000.000000,2792.151971,-1249.084024,579.000000
75%,0.598333,0.647012,20000.000000,5844.601160,-350.760691,737.000000
max,0.967616,0.766419,40000.000000,25563.948363,2347.283665,974.000000


## Expected Loss and Risk-Adjusted Return Distribution

Expected loss and risk-adjusted return are reviewed separately because they directly influence credit decisioning.

A loan may have a low default probability but still produce poor risk-adjusted return if the exposure or potential loss is high.

In [8]:
df[['expected_loss','risk_adjusted_return']].describe()

,expected_loss,risk_adjusted_return
count,40000.000000,40000.000000
mean,4163.416110,-2160.433774
std,3908.355002,2526.859363
min,33.600831,-17734.927705
25%,1317.465334,-3214.107339
50%,2792.151971,-1249.084024
75%,5844.601160,-350.760691
max,25563.948363,2347.283665


## Review Risk Band Categories

The unique risk band values are checked to confirm the available borrower risk segments.

These risk bands are used as one of the inputs for the loan decision rule.

In [9]:
df['risk_band'].unique()

['High', 'Low', 'Very High', 'Medium', 'Very Low']
Categories (5, object): ['Very Low' < 'Low' < 'Medium' < 'High' < 'Very High']

## Create Loan Decision Rule

A rule-based loan decision function is created using risk band, expected loss, and risk-adjusted return.

Decision logic:

- Approve:
  - Very Low or Low risk band
  - Expected loss below threshold
  - Positive risk-adjusted return

- Manual Review:
  - Medium risk band
  - Expected loss below manual review threshold

- Reject:
  - High-risk bands
  - Negative or unattractive risk-adjusted return
  - Expected loss above acceptable threshold

This is a simplified decisioning framework. In real lending environments, additional policy rules, compliance checks, affordability tests, and manual underwriting guidelines would also be applied.

In [10]:
def loan_decision(row):
    if (row['risk_band'] in ['Very Low', 'Low'] and row['expected_loss'] <= 20000000 and row['risk_adjusted_return']>0):
        return 'Approve'
    elif (row['risk_band']== 'Medium' and row['expected_loss'] <= 50000000):
        return 'Manual Review'
    else:
        return 'Reject'

## Apply Loan Decision Rules

The loan decision function is applied to every borrower record.

Each borrower is assigned one of three preliminary decisions:

- Approve
- Manual Review
- Reject

## Loan Decision Distribution

The distribution of loan decisions is reviewed to understand how many borrowers fall into each category.

This helps evaluate whether the decision rule is too strict, too lenient, or balanced.

In [11]:
df['loan_decision']=df.apply(loan_decision, axis=1)
df['loan_decision'].value_counts()

loan_decision
Reject           28201
Manual Review     8000
Approve           3799
Name: count, dtype: int64

## Decision Summary

A summary table is created at the loan decision level.

The summary includes:

- borrower count
- actual default rate
- average LGD
- average EAD
- total expected loss
- average expected loss
- total risk-adjusted return
- average risk-adjusted return
- expected interest income

This helps evaluate the risk and profitability profile of each decision group.

In [15]:
decision_summary = df.groupby('loan_decision').agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
        total_expected_interest_income = ('expected_interest_income','sum'),
        avg_expected_interest_income = ('expected_interest_income','mean'),
    
).reset_index()

decision_summary

,loan_decision,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return,total_expected_interest_income,avg_expected_interest_income
0,Approve,3799,0.039221,2069.072537,0.544636,52302225,13767.366412,3.520841e+06,926.780870,8.843959e+05,232.797023,4.405236e+06,1159.577894
1,Manual Review,8000,0.184125,4790.797142,0.598850,108561100,13570.137500,2.753516e+07,3441.894746,-1.297154e+07,-1621.442993,1.456361e+07,1820.451753
2,Reject,28201,0.256658,17057.130034,0.604841,419678975,14881.705436,1.354806e+08,4804.107866,-7.433020e+07,-2635.729333,6.115044e+07,2168.378533


## Decision Summary by Risk Band

The decision summary is further broken down by both risk band and loan decision.

This helps analyze:

- which risk bands are approved
- which risk bands are rejected
- default rate by risk band and decision
- expected loss by risk band
- risk-adjusted return by risk segment

This view helps validate whether the decisioning framework aligns with borrower risk levels.

In [16]:
decision_summary_risk_band = df.groupby(['risk_band','loan_decision']).agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
).reset_index()

decision_summary_risk_band

/var/folders/99/nqyz4s796bx09s6p71mvxd5h0000gn/T/ipykernel_21489/2955068989.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  decision_summary_risk_band = df.groupby(['risk_band','loan_decision']).agg(


,risk_band,loan_decision,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,Very Low,Approve,3686,0.036896,1998.320944,0.542138,51005475,13837.622084,3.315396e+06,899.456363,8.733628e+05,236.940544
1,Very Low,Manual Review,0,NaN,0.000000,NaN,0,NaN,0.000000e+00,NaN,0.000000e+00,NaN
2,Very Low,Reject,4314,0.058646,2335.766059,0.541439,57457475,13318.839824,5.717546e+06,1325.346789,-1.107841e+06,-256.801378
3,Low,Approve,113,0.115044,70.751593,0.626120,1296750,11475.663717,2.054444e+05,1818.091789,1.103305e+04,97.637569
4,Low,Manual Review,0,NaN,0.000000,NaN,0,NaN,0.000000e+00,NaN,0.000000e+00,NaN
5,Low,Reject,7887,0.107392,4501.282124,0.570722,100042475,12684.477621,1.689645e+07,2142.317012,-5.730041e+06,-726.517215
6,Medium,Approve,0,NaN,0.000000,NaN,0,NaN,0.000000e+00,NaN,0.000000e+00,NaN
7,Medium,Manual Review,8000,0.184125,4790.797142,0.598850,108561100,13570.137500,2.753516e+07,3441.894746,-1.297154e+07,-1621.442993
8,Medium,Reject,0,NaN,0.000000,NaN,0,NaN,0.000000e+00,NaN,0.000000e+00,NaN
9,High,Approve,0,NaN,0.000000,NaN,0,NaN,0.000000e+00,NaN,0.000000e+00,NaN


## Save Decision Engine Outputs

The final decision engine output and decision summary are saved for downstream reporting and analysis.

Saved outputs include:

- borrower-level decision engine output
- decision summary table

These outputs can be used for portfolio monitoring, lending strategy review, and risk-based decision reporting.

In [17]:
df.to_pickle("../data/decision_engine_output.pkl")
decision_summary.to_csv("../data/decision_summary.csv", index=False)


# Conclusion

This notebook created a simplified loan decision engine using machine learning-based credit risk outputs.

Key steps completed:

1. Imported borrower-level PD, LGD, EAD, and Expected Loss data.
2. Reviewed key credit risk metrics.
3. Created rule-based approval logic.
4. Assigned borrowers into Approve, Manual Review, and Reject categories.
5. Summarized risk and profitability by decision group.
6. Analyzed decisions across risk bands.
7. Saved final decision engine outputs.

Key business interpretation:

- Approved borrowers had the lowest actual default rate and positive average risk-adjusted return.
- Manual Review borrowers represented moderate risk and required further underwriting review.
- Rejected borrowers had higher expected loss and weaker risk-adjusted profitability.

This notebook demonstrates how machine learning risk scores can be converted into practical lending decision rules for credit risk management.